<a href="https://colab.research.google.com/github/TheSuhaniVerma/Transfer-Learning-for-URL/blob/main/CODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_PATH = "/content/drive/MyDrive/URL_detection_using_DNN"
DATA_PATH = BASE_PATH + "/DATA"
PLOT_PATH = BASE_PATH + "/PLOTS"
LOG_PATH = BASE_PATH + "/LOG"

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Tranco (benign)
tranco = pd.read_csv(DATA_PATH + "/tranco.csv", header=None)
tranco.columns = ["rank", "url"]
tranco['label'] = 0
tranco = tranco[['url', 'label']]

# PhishTank (malicious)
phish = pd.read_csv(DATA_PATH + "/phishtank.csv")
phish = phish[['url']]
phish['label'] = 1

In [ ]:
df = pd.concat([tranco, phish], ignore_index=True)
df['url'] = df['url'].astype(str)

In [ ]:
import re
import idna
import unicodedata


def preprocess_url(url):

    url = str(url).lower()

    url = re.sub(r"http[s]?://", "", url)
    url = re.sub(r"www\.", "", url)

    url = url.strip()

    try:
        parts = url.split("/",1)
        domain = parts[0]
        path = parts[1] if len(parts)>1 else ""

        decoded_domain = idna.decode(domain)

        if path:
            url = decoded_domain + "/" + path
        else:
            url = decoded_domain

    except:
        pass

    url = unicodedata.normalize("NFKC", url)

    return url

def is_multilingual(url):
    return any(ord(c) > 127 for c in url)


In [ ]:
df['url'] = df['url'].apply(preprocess_url)

df = df.drop_duplicates(subset='url').reset_index(drop=True)

df['is_multilingual'] = df['url'].apply(is_multilingual)

In [ ]:
english_df = df[df['is_multilingual'] == False]
multi_df = df[df['is_multilingual'] == True]

In [ ]:
print(multi_df['label'].value_counts())

label
0    1626
1      77
Name: count, dtype: int64


In [ ]:
confusables = {
    "a": ["а"],   # Cyrillic
    "e": ["е"],
    "o": ["ο"],
    "p": ["р"],
    "c": ["с"],
    "x": ["х"],
    "y": ["у"],
    "m": ["м"],
    "n": ["п"]
}

import random

def generate_homograph(domain):

    new_domain = ""

    for char in domain:
        if char in confusables and random.random() < 0.4:
            new_domain += random.choice(confusables[char])
        else:
            new_domain += char

    return new_domain

In [ ]:
top_domains = tranco["url"].head(1500)
homograph_domains = []

for domain in top_domains:
    fake_domain = generate_homograph(domain)

    if fake_domain != domain:
        homograph_domains.append(fake_domain)

homograph_df = pd.DataFrame({
    "url": homograph_domains,
    "label": 1
})

In [ ]:
multi_df = pd.concat([multi_df, homograph_df], ignore_index=True)

In [ ]:
print(multi_df['url'].head(10))

0    หนังโป๊เด้อ.net
1        ångströ.com
2         求人ボックス.com
3        نسوانجي.net
4      рус-порно.net
5     ทองคําราคา.com
6      ดูคลิปเย็ด.cc
7        หีเด้งง.com
8         หีหอม2.com
9      หนังโป๊จ้า.cc
Name: url, dtype: object


In [ ]:
multi_df = multi_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
print(multi_df['label'].value_counts())

label
0    1626
1    1437
Name: count, dtype: int64


In [ ]:
## balancing eng dataset
benign_en = english_df[english_df.label == 0]
malicious_en = english_df[english_df.label == 1]

size = min(len(benign_en), len(malicious_en))

benign_sample = benign_en.sample(size, random_state=42)
malicious_sample = malicious_en.sample(size, random_state=42)

balanced_eng_df = pd.concat([benign_sample, malicious_sample])

In [ ]:
print(balanced_eng_df['label'].value_counts())

label
0    55144
1    55144
Name: count, dtype: int64


# without Transfer Learning

Data Preparation

In [ ]:
from sklearn.model_selection import train_test_split

# Train on English only
X = balanced_eng_df["url"]
y = balanced_eng_df["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Test on multilingual
X_test = multi_df["url"]
y_test = multi_df["label"]

In [ ]:
print(y_train.value_counts())
print(y_val.value_counts())
print(y_test.value_counts())

label
0    44115
1    44115
Name: count, dtype: int64
label
0    11029
1    11029
Name: count, dtype: int64
label
0    1626
1    1437
Name: count, dtype: int64


In [ ]:
## convert to string

X_train = X_train.astype(str)
X_val = X_val.astype(str)
X_test = X_test.astype(str)

In [ ]:
import pandas as pd

train_df = pd.DataFrame({
    "url": X_train,
    "label": y_train
})

val_df = pd.DataFrame({
    "url": X_val,
    "label": y_val
})

test_df = pd.DataFrame({
    "url": X_test,
    "label": y_test
})

In [ ]:
#saving dataset
train_df.to_csv(f"{DATA_PATH}/train.csv", index=False)
val_df.to_csv(f"{DATA_PATH}/val.csv", index=False)
test_df.to_csv(f"{DATA_PATH}/test.csv", index=False)

print("Datasets saved successfully!")

Datasets saved successfully!


In [ ]:
full_df = pd.concat([train_df, val_df, test_df])

full_df.to_csv(f"{DATA_PATH}/full_dataset.csv", index=False)

In [ ]:
#Tokenization
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(char_level=True, lower=True)

tokenizer.fit_on_texts(list(X_train))

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary size:", vocab_size)

Vocabulary size: 66


In [ ]:
## sequences

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

In [ ]:
#Pading
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 200

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post")
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAX_LEN, padding="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding="post")

In [ ]:
# Features
np.save(f"{DATA_PATH}/X_train_pad.npy", X_train_pad)
np.save(f"{DATA_PATH}/X_val_pad.npy", X_val_pad)
np.save(f"{DATA_PATH}/X_test_pad.npy", X_test_pad)

# Labels
np.save(f"{DATA_PATH}/y_train.npy", y_train)
np.save(f"{DATA_PATH}/y_val.npy", y_val)
np.save(f"{DATA_PATH}/y_test.npy", y_test)

print("Tokenized data saved!")

Tokenized data saved!


In [ ]:
import pickle

with open(f"{DATA_PATH}/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved!")

Tokenizer saved!


In [ ]:
with open(f"{DATA_PATH}/token_info.txt", "w") as f:
    f.write(f"Vocabulary Size: {vocab_size}\n")
    f.write(f"Max Length: {MAX_LEN}\n")

PLOTS

In [ ]:
import os

def get_model_path(model_name):
    path = f"{PLOT_PATH}/{model_name.upper()}"
    os.makedirs(path, exist_ok=True)
    return path

In [ ]:
import datetime

os.makedirs(LOG_PATH, exist_ok=True)

def save_metrics(history, model_name):

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = f"{LOG_PATH}/{model_name}_{timestamp}.txt"

    with open(file_path, "w") as f:
        f.write(f"Model: {model_name.upper()} ")
        f.write(f"Run Time: {timestamp}\n\n")

        epochs = len(history.history['loss'])

        for i in range(epochs):
            f.write(f"Epoch {i+1}\n")
            f.write(f"Train Loss: {history.history['loss'][i]:.4f}\n")
            f.write(f"Val Loss: {history.history['val_loss'][i]:.4f}\n")
            f.write(f"Train Acc: {history.history['accuracy'][i]*100:.2f}%\n")
            f.write(f"Val Acc: {history.history['val_accuracy'][i]*100:.2f}%\n")
            f.write("-"*40 + "\n")

    print(f"Saved metrics → {file_path}")

In [ ]:
def save_train_plots(history, model_name):

    path = get_model_path(model_name)
    epochs = range(1, len(history.history['loss']) + 1)

    # 🔸 Accuracy
    plt.figure()
    plt.plot(epochs, [x*100 for x in history.history['accuracy']], 'bo-', label='Train')
    plt.plot(epochs, [x*100 for x in history.history['val_accuracy']], 'r*-', label='Val')
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy (%)")
    plt.title(f"{model_name.upper()} Accuracy")
    plt.legend()
    plt.grid(True)

    plt.savefig(f"{path}/{model_name}_accuracy.png",
                dpi=80, bbox_inches='tight', pad_inches=0.1)
    plt.savefig(f"{path}/{model_name}_accuracy.pdf",
                dpi=80, bbox_inches='tight', pad_inches=0.1)
    plt.close()


    # 🔸 Loss
    plt.figure()
    plt.plot(epochs, history.history['loss'], 'bo-', label='Train')
    plt.plot(epochs, history.history['val_loss'], 'r*-', label='Val')
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title(f"{model_name.upper()} Loss")
    plt.legend()
    plt.grid(True)

    plt.savefig(f"{path}/{model_name}_loss.png",
                dpi=80, bbox_inches='tight', pad_inches=0.1)
    plt.savefig(f"{path}/{model_name}_loss.pdf",
                dpi=80, bbox_inches='tight', pad_inches=0.1)
    plt.close()

In [ ]:
def plot_comparison(model_name, history, model, X_test, y_test):
    path = get_model_path(model_name)
    mod = model_name.upper()

    val_acc = history.history['val_accuracy'][-1] * 100
    multi_acc = model.evaluate(X_test, y_test, verbose=0)[1] * 100

    plt.figure()
    plt.bar(["English", "Multilingual"], [val_acc, multi_acc])

    plt.ylabel("Accuracy (%)")
    plt.title(f"{mod} Performance Comparison ")

    plt.savefig(f"{path}/{model_name}_performance.png",
                dpi=80, bbox_inches='tight', pad_inches=0.1)


    plt.close()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

def plot_confusion_matrix(model_name, model, X_test, y_test):
    path = get_model_path(model_name)
    y_pred = (model.predict(X_test) > 0.5).astype(int)

    cm = confusion_matrix(y_test, y_pred)

    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"{model_name.upper()} Confusion Matrix")

    plt.savefig(f"{path}/{model_name}_confusion.png",
                dpi=80, bbox_inches='tight', pad_inches=0.1)


    plt.close()

In [ ]:
def plot_tl_training(history_pre, history_ft, model_name):

    path = get_model_path(model_name)

    # Combine epochs
    train_acc = history_pre.history['accuracy'] + history_ft.history['accuracy']
    val_acc   = history_pre.history['val_accuracy'] + history_ft.history['val_accuracy']

    train_loss = history_pre.history['loss'] + history_ft.history['loss']
    val_loss   = history_pre.history['val_loss'] + history_ft.history['val_loss']

    epochs = range(1, len(train_acc) + 1)
    split_epoch = len(history_pre.history['accuracy'])

    # 🔹 Accuracy Plot
    plt.figure(figsize=(8,5))
    plt.plot(epochs, [x*100 for x in train_acc], label="Train")
    plt.plot(epochs, [x*100 for x in val_acc], label="Validation")

    # Mark boundary
    plt.axvline(x=split_epoch, color='black', linestyle='--')
    plt.text(split_epoch+0.2, max(train_acc)*100, "Fine-tuning", fontsize=9)

    plt.xlabel("Epochs")
    plt.ylabel("Accuracy (%)")
    plt.title(f"{model_name.upper()} Training (Pretrain + Finetune)")
    plt.legend()
    plt.grid(True)

    plt.savefig(f"{path}/{model_name}_TL_accuracy.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{path}/{model_name}_TL_accuracy.pdf", dpi=300, bbox_inches='tight')
    plt.close()

    # 🔹 Loss Plot
    plt.figure(figsize=(8,5))
    plt.plot(epochs, train_loss, label="Train")
    plt.plot(epochs, val_loss, label="Validation")

    plt.axvline(x=split_epoch, color='black', linestyle='--')
    plt.text(split_epoch+0.2, max(train_loss), "Fine-tuning", fontsize=9)

    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title(f"{model_name.upper()} Training (Pretrain + Finetune)")
    plt.legend()
    plt.grid(True)

    plt.savefig(f"{path}/{model_name}_TL_loss.png", dpi=80, bbox_inches='tight',pad_inches=0.1)
    plt.savefig(f"{path}/{model_name}_TL_loss.pdf", dpi=80, bbox_inches='tight',pad_inches=0.1)
    plt.close()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_all_models(models_data, X_en, y_en, X_multi, y_multi):

    labels = [
        "CNN", "LSTM", "GRU",
        "CNN+LSTM", "CNN+GRU",
        "CNN+TL", "LSTM+TL", "GRU+TL",
        "CNN+LSTM+TL", "CNN+GRU+TL"
    ]

    en_acc = []
    multi_acc = []

    for name, history, model in models_data:
        # Evaluate properly on both datasets
        en_score = model.evaluate(X_en, y_en, verbose=0)[1] * 100
        multi_score = model.evaluate(X_multi, y_multi, verbose=0)[1] * 100

        en_acc.append(en_score)
        multi_acc.append(multi_score)

    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(12, 6))

    bars1 = plt.bar(x - width/2, en_acc, width, label="English URL")
    bars2 = plt.bar(x + width/2, multi_acc, width, label="Multilingual URL")

    # Rotate labels for clarity
    plt.xticks(x, labels, rotation=45, ha='right')

    plt.ylabel("Accuracy (%)")
    plt.title("Model Comparison")

    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.6)

    # Optional: add values on top of bars
    for bar in bars1:
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5,
                 f"{bar.get_height():.1f}", ha='center', fontsize=8)

    for bar in bars2:
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5,
                 f"{bar.get_height():.1f}", ha='center', fontsize=8)

    plt.tight_layout()

    plt.savefig(f"{PLOT_PATH}/all_models_comparison_fixed.png", dpi=120)
    plt.savefig(f"{PLOT_PATH}/all_models_comparison_fixed.pdf", dpi=120)

    plt.close()

    print("✅ Fixed comparison plot saved!")

In [ ]:
import matplotlib.pyplot as plt

def save_results_table(results_df, save_path):

    # Round values for clean display
    df = results_df.copy()
    for col in ["Accuracy", "Precision", "Recall", "F1"]:
        df[col] = df[col].round(3)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.axis('off')

    # Create table
    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc='center',
        loc='center'
    )

    # Styling
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.auto_set_column_width(col=list(range(len(df.columns))))
    table.scale(1, 1.5)

    # Bold header
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold')

    plt.title("Model Performance Comparison", fontsize=12, pad=10)

    # Save high-quality
    plt.savefig(f"{save_path}/results_table.png", dpi=80, bbox_inches='tight',pad_inches=0.1)
    plt.savefig(f"{save_path}/results_table.pdf", dpi=80, bbox_inches='tight',pad_inches=0.1)

    plt.close()

MODELS

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

EMBED_DIM = 64

cnn_model = Sequential()

cnn_model.add(Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN))
cnn_model.add(Conv1D(128, 5, activation='relu'))
cnn_model.add(GlobalMaxPooling1D())
cnn_model.add(Dense(64, activation='relu'))
cnn_model.add(Dropout(0.5))
cnn_model.add(Dense(1, activation='sigmoid'))

cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

lstm_model = Sequential()

lstm_model.add(Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN))
lstm_model.add(LSTM(128))
lstm_model.add(Dense(64, activation='relu'))
lstm_model.add(Dropout(0.5))
lstm_model.add(Dense(1, activation='sigmoid'))

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.layers import GRU

gru_model = Sequential()

gru_model.add(Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN))
gru_model.add(GRU(128))
gru_model.add(Dense(64, activation='relu'))
gru_model.add(Dropout(0.5))
gru_model.add(Dense(1, activation='sigmoid'))

gru_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# CNN_LSTM Hybrid
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout

cnn_lstm_model = Sequential()

cnn_lstm_model.add(Embedding(vocab_size, 64, input_length=MAX_LEN))
cnn_lstm_model.add(Conv1D(128, 5, activation='relu'))
cnn_lstm_model.add(MaxPooling1D(pool_size=2))
cnn_lstm_model.add(LSTM(64))
cnn_lstm_model.add(Dense(64, activation='relu'))
cnn_lstm_model.add(Dropout(0.3))
cnn_lstm_model.add(Dense(1, activation='sigmoid'))

cnn_lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# CNN_GRU hybrid
from tensorflow.keras.layers import GRU

cnn_gru_model = Sequential()

cnn_gru_model.add(Embedding(vocab_size, 64, input_length=MAX_LEN))
cnn_gru_model.add(Conv1D(128, 5, activation='relu'))
cnn_gru_model.add(MaxPooling1D(pool_size=2))
cnn_gru_model.add(GRU(64))
cnn_gru_model.add(Dense(64, activation='relu'))
cnn_gru_model.add(Dropout(0.3))
cnn_gru_model.add(Dense(1, activation='sigmoid'))

cnn_gru_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_cnn = cnn_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs = 10,
    batch_size=64
)

Epoch 1/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - accuracy: 0.9775 - loss: 0.0832 - val_accuracy: 0.9869 - val_loss: 0.0515
Epoch 2/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9871 - loss: 0.0538 - val_accuracy: 0.9890 - val_loss: 0.0485
Epoch 3/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9882 - loss: 0.0488 - val_accuracy: 0.9872 - val_loss: 0.0514
Epoch 4/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.9885 - loss: 0.0453 - val_accuracy: 0.9886 - val_loss: 0.0482
Epoch 5/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9897 - loss: 0.0428 - val_accuracy: 0.9881 - val_loss: 0.0491
Epoch 6/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9901 - loss: 0.0402 - val_accuracy: 0.9883 - val_loss: 0.0524
Epoch 7/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9900 - loss: 0.0377 - val_accuracy: 0.9885 - val_loss: 0.0492
Epoch 8/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9907 - loss: 0.0345 

In [ ]:
history_lstm = lstm_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - accuracy: 0.7410 - loss: 0.4038 - val_accuracy: 0.9844 - val_loss: 0.0727
Epoch 2/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 20s 15ms/step - accuracy: 0.9829 - loss: 0.0821 - val_accuracy: 0.9852 - val_loss: 0.0694
Epoch 3/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.9852 - loss: 0.0729 - val_accuracy: 0.9856 - val_loss: 0.0673
Epoch 4/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 20s 15ms/step - accuracy: 0.9855 - loss: 0.0713 - val_accuracy: 0.9860 - val_loss: 0.0651
Epoch 5/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.9860 - loss: 0.0667 - val_accuracy: 0.9861 - val_loss: 0.0556
Epoch 6/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 20s 14ms/step - accuracy: 0.9860 - loss: 0.0609 - val_accuracy: 0.9862 - val_loss: 0.0547
Epoch 7/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 20s 14ms/step - accuracy: 0.9864 - loss: 0.0559 - val_accuracy: 0.9852 - val_loss: 0.0542
Epoch 8/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 20s 14ms/step - accuracy: 0.9868 -

In [ ]:
history_gru = gru_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - accuracy: 0.7158 - loss: 0.4351 - val_accuracy: 0.9832 - val_loss: 0.0723
Epoch 2/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.9842 - loss: 0.0728 - val_accuracy: 0.9855 - val_loss: 0.0563
Epoch 3/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.9858 - loss: 0.0643 - val_accuracy: 0.9865 - val_loss: 0.0541
Epoch 4/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.9865 - loss: 0.0598 - val_accuracy: 0.9862 - val_loss: 0.0527
Epoch 5/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.9871 - loss: 0.0564 - val_accuracy: 0.9874 - val_loss: 0.0504
Epoch 6/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 20s 14ms/step - accuracy: 0.9875 - loss: 0.0521 - val_accuracy: 0.9871 - val_loss: 0.0510
Epoch 7/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.9878 - loss: 0.0512 - val_accuracy: 0.9880 - val_loss: 0.0479
Epoch 8/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.9881 -

In [ ]:
history_cnn_lstm = cnn_lstm_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - accuracy: 0.9171 - loss: 0.1685 - val_accuracy: 0.9862 - val_loss: 0.0598
Epoch 2/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.9857 - loss: 0.0601 - val_accuracy: 0.9865 - val_loss: 0.0551
Epoch 3/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - accuracy: 0.9870 - loss: 0.0538 - val_accuracy: 0.9877 - val_loss: 0.0500
Epoch 4/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - accuracy: 0.9880 - loss: 0.0499 - val_accuracy: 0.9854 - val_loss: 0.0542
Epoch 5/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - accuracy: 0.9884 - loss: 0.0478 - val_accuracy: 0.9889 - val_loss: 0.0475
Epoch 6/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - accuracy: 0.9890 - loss: 0.0454 - val_accuracy: 0.9865 - val_loss: 0.0539
Epoch 7/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.9896 - loss: 0.0426 - val_accuracy: 0.9887 - val_loss: 0.0468
Epoch 8/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - accuracy: 0.9898 -

In [ ]:
history_cnn_gru = cnn_gru_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - accuracy: 0.9107 - loss: 0.1694 - val_accuracy: 0.9862 - val_loss: 0.0559
Epoch 2/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - accuracy: 0.9869 - loss: 0.0550 - val_accuracy: 0.9878 - val_loss: 0.0495
Epoch 3/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - accuracy: 0.9878 - loss: 0.0507 - val_accuracy: 0.9874 - val_loss: 0.0500
Epoch 4/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - accuracy: 0.9885 - loss: 0.0475 - val_accuracy: 0.9880 - val_loss: 0.0493
Epoch 5/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - accuracy: 0.9892 - loss: 0.0457 - val_accuracy: 0.9887 - val_loss: 0.0485
Epoch 6/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 15s 11ms/step - accuracy: 0.9897 - loss: 0.0433 - val_accuracy: 0.9886 - val_loss: 0.0472
Epoch 7/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 15s 11ms/step - accuracy: 0.9898 - loss: 0.0414 - val_accuracy: 0.9890 - val_loss: 0.0462
Epoch 8/10
1379/1379 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - accuracy: 0.9907 -

In [ ]:
print("CNN Test Acc:", cnn_model.evaluate(X_test_pad, y_test)[1])
print("LSTM Test Acc:", lstm_model.evaluate(X_test_pad, y_test)[1])
print("GRU Test Acc:", gru_model.evaluate(X_test_pad, y_test)[1])
print("CNN+LSTM:", cnn_lstm_model.evaluate(X_test_pad, y_test)[1])
print("CNN+GRU:", cnn_gru_model.evaluate(X_test_pad, y_test)[1])

96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5560 - loss: 3.3115
CNN Test Acc: 0.5559908747673035
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5553 - loss: 2.1430
LSTM Test Acc: 0.5553379058837891
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5550 - loss: 2.2015
GRU Test Acc: 0.5550114512443542
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5553 - loss: 2.2803
CNN+LSTM: 0.5553379058837891
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5547 - loss: 2.5371
CNN+GRU: 0.5546849370002747


In [ ]:
# CNN
save_metrics(history_cnn, "cnn")
save_train_plots(history_cnn, "cnn")
plot_comparison("cnn", history_cnn, cnn_model, X_test_pad, y_test)
plot_confusion_matrix("cnn", cnn_model, X_test_pad, y_test)

# LSTM
save_metrics(history_lstm, "lstm")
save_train_plots(history_lstm, "lstm")
plot_comparison("lstm", history_lstm, lstm_model, X_test_pad, y_test)
plot_confusion_matrix("lstm", lstm_model, X_test_pad, y_test)

# GRU
save_metrics(history_gru, "gru")
save_train_plots(history_gru, "gru")
plot_comparison("gru", history_gru, gru_model, X_test_pad, y_test)
plot_confusion_matrix("gru", gru_model, X_test_pad, y_test)

# CNN-LSTM
save_metrics(history_cnn_lstm, "cnn_lstm")
save_train_plots(history_cnn_lstm, "cnn_lstm")
plot_comparison("cnn_lstm", history_cnn_lstm, cnn_lstm_model, X_test_pad, y_test)
plot_confusion_matrix("cnn_lstm", cnn_lstm_model, X_test_pad, y_test)

# CNN-GRU
save_metrics(history_cnn_gru, "cnn_gru")
save_train_plots(history_cnn_gru, "cnn_gru")
plot_comparison("cnn_gru", history_cnn_gru, cnn_gru_model, X_test_pad, y_test)
plot_confusion_matrix("cnn_gru", cnn_gru_model, X_test_pad, y_test)

Saved metrics → /content/drive/MyDrive/URL_detection_using_DNN/LOG/cnn_20260429_114333.txt
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Saved metrics → /content/drive/MyDrive/URL_detection_using_DNN/LOG/lstm_20260429_114336.txt
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Saved metrics → /content/drive/MyDrive/URL_detection_using_DNN/LOG/gru_20260429_114338.txt
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
Saved metrics → /content/drive/MyDrive/URL_detection_using_DNN/LOG/cnn_lstm_20260429_114341.txt
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
Saved metrics → /content/drive/MyDrive/URL_detection_using_DNN/LOG/cnn_gru_20260429_114347.txt
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


# Transfer Learning

Data preparation

In [ ]:
X_pre = balanced_eng_df["url"]
y_pre = balanced_eng_df["label"]

X_ft = multi_df["url"]
y_ft = multi_df["label"]

In [ ]:
#pretraining dataset
from sklearn.model_selection import train_test_split

X_train_pre, X_val_pre, y_train_pre, y_val_pre = train_test_split(
    X_pre,
    y_pre,
    test_size=0.1,
    stratify=y_pre,
    random_state=42
)

In [ ]:
#finetuning dataset
X_train_ft, X_test_ft, y_train_ft, y_test_ft = train_test_split(
    X_ft,
    y_ft,
    test_size=0.2,
    stratify=y_ft,
    random_state=42
)

In [ ]:
X_train_pre = X_train_pre.astype(str)
X_val_pre = X_val_pre.astype(str)

X_train_ft = X_train_ft.astype(str)
X_test_ft = X_test_ft.astype(str)

In [ ]:
print("Pretrain train:", len(X_train_pre))
print("Pretrain val:", len(X_val_pre))
print("Finetune train:", len(X_train_ft))
print("Finetune test:", len(X_test_ft))

Pretrain train: 99259
Pretrain val: 11029
Finetune train: 2450
Finetune test: 613


tokenization

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizerTL = Tokenizer(char_level=True, lower=False)

tokenizerTL.fit_on_texts(
    list(X_train_pre) + list(X_train_ft)
)

vocab_size = len(tokenizerTL.word_index) + 1
print("Vocab size:", vocab_size)

Vocab size: 992


In [ ]:
# Pretraining
X_train_pre_seq = tokenizerTL.texts_to_sequences(X_train_pre)
X_val_pre_seq   = tokenizerTL.texts_to_sequences(X_val_pre)

# Fine-tuning
X_train_ft_seq  = tokenizerTL.texts_to_sequences(X_train_ft)
X_test_ft_seq   = tokenizerTL.texts_to_sequences(X_test_ft)

Padding

In [ ]:
#Pading
from tensorflow.keras.preprocessing.sequence import pad_sequences
MAX_LEN = 200

X_train_pre_pad = pad_sequences(X_train_pre_seq, maxlen=MAX_LEN, padding="post")
X_val_pre_pad   = pad_sequences(X_val_pre_seq, maxlen=MAX_LEN, padding="post")

X_train_ft_pad  = pad_sequences(X_train_ft_seq, maxlen=MAX_LEN, padding="post")
X_test_ft_pad   = pad_sequences(X_test_ft_seq, maxlen=MAX_LEN, padding="post")

Models

In [ ]:
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [ ]:
#CNN
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# PRETRAIN
cnn_TL_model = Sequential()
cnn_TL_model.add(Embedding(vocab_size, 128,input_length=MAX_LEN))
cnn_TL_model.add(Conv1D(128, 5, activation='relu'))
cnn_TL_model.add(GlobalMaxPooling1D())
cnn_TL_model.add(Dense(64, activation='relu'))
cnn_TL_model.add(Dropout(0.5))
cnn_TL_model.add(Dense(1, activation='sigmoid'))

cnn_TL_model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])

history_cnn_pre = cnn_TL_model.fit(
    X_train_pre_pad, y_train_pre,
    epochs=10, batch_size=128,
    validation_data=(X_val_pre_pad, y_val_pre)
)




cnn_TL_model.compile(optimizer=Adam(0.0005), loss='binary_crossentropy', metrics=['accuracy'])

# FINETUNE
history_cnn_TL = cnn_TL_model.fit(
    X_train_ft_pad, y_train_ft,
    epochs=5, batch_size=64,
    validation_data=(X_test_ft_pad, y_test_ft)
)

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


776/776 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.9765 - loss: 0.0858 - val_accuracy: 0.9853 - val_loss: 0.0552
Epoch 2/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9872 - loss: 0.0526 - val_accuracy: 0.9880 - val_loss: 0.0501
Epoch 3/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9887 - loss: 0.0481 - val_accuracy: 0.9879 - val_loss: 0.0512
Epoch 4/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9887 - loss: 0.0466 - val_accuracy: 0.9873 - val_loss: 0.0527
Epoch 5/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9896 - loss: 0.0437 - val_accuracy: 0.9887 - val_loss: 0.0478
Epoch 6/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9898 - loss: 0.0412 - val_accuracy: 0.9860 - val_loss: 0.0542
Epoch 7/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9903 - loss: 0.0385 - val_accuracy: 0.9883 - val_loss: 0.0526
Epoch 8/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9908 - loss: 0.0363 - val_accuracy: 0.9879 - va

In [ ]:
#LSTM
from tensorflow.keras.layers import LSTM

lstm_TL_model = Sequential()
lstm_TL_model.add(Embedding(vocab_size, 128,input_length=MAX_LEN))
lstm_TL_model.add(LSTM(128))
lstm_TL_model.add(Dense(64, activation='relu'))
lstm_TL_model.add(Dropout(0.5))
lstm_TL_model.add(Dense(1, activation='sigmoid'))

lstm_TL_model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])

history_lstm_pre = lstm_TL_model.fit(
    X_train_pre_pad, y_train_pre,
    epochs=10, batch_size=128,
    validation_data=(X_val_pre_pad, y_val_pre)
)



lstm_TL_model.compile(optimizer=Adam(0.0005), loss='binary_crossentropy', metrics=['accuracy'])

history_lstm_TL = lstm_TL_model.fit(
    X_train_ft_pad, y_train_ft,
    epochs=5, batch_size=64,
    validation_data=(X_test_ft_pad, y_test_ft)
)

Epoch 1/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.6417 - loss: 0.5424 - val_accuracy: 0.9810 - val_loss: 0.0817
Epoch 2/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9840 - loss: 0.0832 - val_accuracy: 0.9825 - val_loss: 0.0731
Epoch 3/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9850 - loss: 0.0710 - val_accuracy: 0.9849 - val_loss: 0.0611
Epoch 4/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9854 - loss: 0.0655 - val_accuracy: 0.9846 - val_loss: 0.0628
Epoch 5/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9860 - loss: 0.0603 - val_accuracy: 0.9851 - val_loss: 0.0553
Epoch 6/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - accuracy: 0.9861 - loss: 0.0592 - val_accuracy: 0.9858 - val_loss: 0.0561
Epoch 7/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - accuracy: 0.9866 - loss: 0.0558 - val_accuracy: 0.9858 - val_loss: 0.0542
Epoch 8/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9869 - loss: 0.0544 - 

In [ ]:
#GRU
from tensorflow.keras.layers import GRU

gru_TL_model = Sequential()
gru_TL_model.add(Embedding(vocab_size, 128,input_length=MAX_LEN))
gru_TL_model.add(GRU(128))
gru_TL_model.add(Dense(64, activation='relu'))
gru_TL_model.add(Dropout(0.5))
gru_TL_model.add(Dense(1, activation='sigmoid'))

gru_TL_model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])

history_gru_pre = gru_TL_model.fit(
    X_train_pre_pad, y_train_pre,
    epochs=10, batch_size=128,
    validation_data=(X_val_pre_pad, y_val_pre)
)

gru_TL_model.compile(optimizer=Adam(0.0005), loss='binary_crossentropy', metrics=['accuracy'])

history_gru_TL = gru_TL_model.fit(
    X_train_ft_pad, y_train_ft,
    epochs=5, batch_size=64,
    validation_data=(X_test_ft_pad, y_test_ft)
)

Epoch 1/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.5704 - loss: 0.6278 - val_accuracy: 0.9727 - val_loss: 0.0989
Epoch 2/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.9827 - loss: 0.0759 - val_accuracy: 0.9851 - val_loss: 0.0568
Epoch 3/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.9857 - loss: 0.0616 - val_accuracy: 0.9859 - val_loss: 0.0548
Epoch 4/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9863 - loss: 0.0574 - val_accuracy: 0.9872 - val_loss: 0.0522
Epoch 5/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.9866 - loss: 0.0563 - val_accuracy: 0.9860 - val_loss: 0.0527
Epoch 6/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.9869 - loss: 0.0543 - val_accuracy: 0.9879 - val_loss: 0.0503
Epoch 7/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.9876 - loss: 0.0512 - val_accuracy: 0.9862 - val_loss: 0.0519
Epoch 8/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.9880 - loss: 0.0502 - 

In [ ]:
#CNN-LSTM
cnn_lstm_TL_model = Sequential()
cnn_lstm_TL_model.add(Embedding(vocab_size, 128,input_length=MAX_LEN))
cnn_lstm_TL_model.add(Conv1D(128, 5, activation='relu'))
cnn_lstm_TL_model.add(LSTM(128))
cnn_lstm_TL_model.add(Dense(64, activation='relu'))
cnn_lstm_TL_model.add(Dropout(0.5))
cnn_lstm_TL_model.add(Dense(1, activation='sigmoid'))

cnn_lstm_TL_model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])

history_cnn_lstm_pre = cnn_lstm_TL_model.fit(
    X_train_pre_pad, y_train_pre,
    epochs=10, batch_size=128,
    validation_data=(X_val_pre_pad, y_val_pre)
)



cnn_lstm_TL_model.compile(optimizer=Adam(0.0005), loss='binary_crossentropy', metrics=['accuracy'])

history_cnn_lstm_TL = cnn_lstm_TL_model.fit(
    X_train_ft_pad, y_train_ft,
    epochs=5, batch_size=64,
    validation_data=(X_test_ft_pad, y_test_ft)
)

Epoch 1/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 20s 23ms/step - accuracy: 0.8618 - loss: 0.2490 - val_accuracy: 0.9850 - val_loss: 0.0610
Epoch 2/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.9856 - loss: 0.0643 - val_accuracy: 0.9859 - val_loss: 0.0544
Epoch 3/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.9871 - loss: 0.0573 - val_accuracy: 0.9859 - val_loss: 0.0528
Epoch 4/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.9876 - loss: 0.0531 - val_accuracy: 0.9875 - val_loss: 0.0551
Epoch 5/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - accuracy: 0.9881 - loss: 0.0512 - val_accuracy: 0.9883 - val_loss: 0.0523
Epoch 6/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.9888 - loss: 0.0491 - val_accuracy: 0.9877 - val_loss: 0.0493
Epoch 7/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.9892 - loss: 0.0468 - val_accuracy: 0.9880 - val_loss: 0.0466
Epoch 8/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.9897 - loss: 0.0438 - 

In [ ]:
#CNN-GRU
cnn_gru_TL_model = Sequential()
cnn_gru_TL_model.add(Embedding(vocab_size, 128,input_length=MAX_LEN))
cnn_gru_TL_model.add(Conv1D(128, 5, activation='relu'))
cnn_gru_TL_model.add(GRU(128))
cnn_gru_TL_model.add(Dense(64, activation='relu'))
cnn_gru_TL_model.add(Dropout(0.5))
cnn_gru_TL_model.add(Dense(1, activation='sigmoid'))

cnn_gru_TL_model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])

history_cnn_gru_pre = cnn_gru_TL_model.fit(
    X_train_pre_pad, y_train_pre,
    epochs=10, batch_size=128,
    validation_data=(X_val_pre_pad, y_val_pre)
)



cnn_gru_TL_model.compile(optimizer=Adam(0.0005), loss='binary_crossentropy', metrics=['accuracy'])

history_cnn_gru_TL = cnn_gru_TL_model.fit(
    X_train_ft_pad, y_train_ft,
    epochs=5, batch_size=64,
    validation_data=(X_test_ft_pad, y_test_ft)
)

Epoch 1/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 19s 22ms/step - accuracy: 0.8495 - loss: 0.2602 - val_accuracy: 0.9747 - val_loss: 0.0823
Epoch 2/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - accuracy: 0.9873 - loss: 0.0586 - val_accuracy: 0.9868 - val_loss: 0.0568
Epoch 3/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.9878 - loss: 0.0544 - val_accuracy: 0.9873 - val_loss: 0.0510
Epoch 4/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.9884 - loss: 0.0507 - val_accuracy: 0.9874 - val_loss: 0.0489
Epoch 5/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.9890 - loss: 0.0482 - val_accuracy: 0.9883 - val_loss: 0.0474
Epoch 6/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.9891 - loss: 0.0463 - val_accuracy: 0.9891 - val_loss: 0.0457
Epoch 7/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.9898 - loss: 0.0431 - val_accuracy: 0.9891 - val_loss: 0.0453
Epoch 8/10
776/776 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.9900 - loss: 0.0410 - 

In [ ]:
print("CNN TL Test Acc:", cnn_TL_model.evaluate(X_test_ft_pad, y_test_ft)[1])
print("LSTM TL Test Acc:", lstm_TL_model.evaluate(X_test_ft_pad, y_test_ft)[1])
print("GRU TL Test Acc:", gru_TL_model.evaluate(X_test_ft_pad, y_test_ft)[1])
print("CNN+LSTM TL:", cnn_lstm_TL_model.evaluate(X_test_ft_pad, y_test_ft)[1])
print("CNN+GRU TL:", cnn_gru_TL_model.evaluate(X_test_ft_pad, y_test_ft)[1])

20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9413 - loss: 0.1377
CNN TL Test Acc: 0.9412724375724792
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9445 - loss: 0.1391
LSTM TL Test Acc: 0.9445350766181946
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9788 - loss: 0.0733
GRU TL Test Acc: 0.9787928462028503
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9494 - loss: 0.1681
CNN+LSTM TL: 0.9494290351867676
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9413 - loss: 0.1531
CNN+GRU TL: 0.9412724375724792


In [ ]:
save_metrics(history_cnn_TL, "cnn_tl")
plot_comparison("cnn_tl",history_cnn_TL,cnn_TL_model,X_test_ft_pad,y_test_ft)
plot_confusion_matrix("cnn_tl",cnn_TL_model,X_test_ft_pad,y_test_ft)


save_metrics(history_lstm_TL, "lstm_tl")
plot_comparison("lstm_tl", history_lstm_TL,lstm_TL_model,X_test_ft_pad,y_test_ft)
plot_confusion_matrix("lstm_tl",lstm_TL_model,X_test_ft_pad,y_test_ft)


save_metrics(history_gru_TL, "gru_tl")
plot_comparison("gru_tl",history_gru_TL,gru_TL_model,X_test_ft_pad,y_test_ft)
plot_confusion_matrix("gru_tl", gru_TL_model, X_test_ft_pad,y_test_ft)


save_metrics(history_cnn_lstm_TL, "cnn_lstm_tl")
plot_comparison("cnn_lstm_tl",history_cnn_lstm_TL,cnn_lstm_TL_model,X_test_ft_pad,y_test_ft)
plot_confusion_matrix("cnn_lstm_tl",cnn_lstm_TL_model,X_test_ft_pad,y_test_ft)


save_metrics(history_cnn_gru_TL, "cnn_gru_tl")
plot_comparison( "cnn_gru_tl",history_cnn_gru_TL,cnn_gru_TL_model,X_test_ft_pad,y_test_ft)
plot_confusion_matrix("cnn_gru_tl",cnn_gru_TL_model,X_test_ft_pad,y_test_ft)

Saved metrics → /content/drive/MyDrive/URL_detection_using_DNN/LOG/cnn_tl_20260429_115658.txt
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
Saved metrics → /content/drive/MyDrive/URL_detection_using_DNN/LOG/lstm_tl_20260429_115659.txt
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
Saved metrics → /content/drive/MyDrive/URL_detection_using_DNN/LOG/gru_tl_20260429_115700.txt
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Saved metrics → /content/drive/MyDrive/URL_detection_using_DNN/LOG/cnn_lstm_tl_20260429_115701.txt
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
Saved metrics → /content/drive/MyDrive/URL_detection_using_DNN/LOG/cnn_gru_tl_20260429_115702.txt
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


In [ ]:
plot_tl_training(history_cnn_pre, history_cnn_TL, "cnn_tl")
plot_tl_training(history_lstm_pre, history_lstm_TL, "lstm_tl")
plot_tl_training(history_gru_pre, history_gru_TL, "gru_tl")
plot_tl_training(history_cnn_lstm_pre, history_cnn_lstm_TL, "cnn_lstm_tl")
plot_tl_training(history_cnn_gru_pre, history_cnn_gru_TL, "cnn_gru_tl")

In [ ]:
plot_all_models(
    [
        ("cnn", history_cnn, cnn_model),
        ("lstm", history_lstm, lstm_model),
        ("gru", history_gru, gru_model),
        ("cnn_lstm", history_cnn_lstm, cnn_lstm_model),
        ("cnn_gru", history_cnn_gru, cnn_gru_model),
        ("cnn_tl", history_cnn_TL, cnn_TL_model),
        ("lstm_tl", history_lstm_TL, lstm_TL_model),
        ("gru_tl", history_gru_TL, gru_TL_model),
        ("cnn_lstm_tl", history_cnn_lstm_TL, cnn_lstm_TL_model),
        ("cnn_gru_tl", history_cnn_gru_TL, cnn_gru_TL_model)
    ],
    X_val_pad, y_val,          # English dataset
    X_test_ft_pad, y_test_ft   # Multilingual dataset
)

NameError: name 'history_cnn' is not defined

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

def get_metrics(model, X, y):
    y_pred = (model.predict(X) > 0.5).astype("int32")

    return {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred),
        "Recall": recall_score(y, y_pred),
        "F1": f1_score(y, y_pred)
    }

results = []

# CNN
results.append({"Model": "CNN Baseline", **get_metrics(cnn_model, X_test_ft_pad, y_test_ft)})
results.append({"Model": "CNN TL", **get_metrics(cnn_TL_model, X_test_ft_pad, y_test_ft)})

# LSTM
results.append({"Model": "LSTM Baseline", **get_metrics(lstm_model, X_test_ft_pad, y_test_ft)})
results.append({"Model": "LSTM TL", **get_metrics(lstm_TL_model, X_test_ft_pad, y_test_ft)})

# GRU
results.append({"Model": "GRU Baseline", **get_metrics(gru_model, X_test_ft_pad, y_test_ft)})
results.append({"Model": "GRU TL", **get_metrics(gru_TL_model, X_test_ft_pad, y_test_ft)})

# CNN-LSTM
results.append({"Model": "CNN-LSTM Baseline", **get_metrics(cnn_lstm_model, X_test_ft_pad, y_test_ft)})
results.append({"Model": "CNN-LSTM TL", **get_metrics(cnn_lstm_TL_model, X_test_ft_pad, y_test_ft)})

# CNN-GRU
results.append({"Model": "CNN-GRU Baseline", **get_metrics(cnn_gru_model, X_test_ft_pad, y_test_ft)})
results.append({"Model": "CNN-GRU TL", **get_metrics(cnn_gru_TL_model, X_test_ft_pad, y_test_ft)})

results_df = pd.DataFrame(results)
results_df

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


,Model,Accuracy,Precision,Recall,F1
0,CNN Baseline,0.556281,0.863636,0.065972,0.122581
1,CNN TL,0.941272,0.953237,0.920139,0.936396
2,LSTM Baseline,0.575856,0.850000,0.118056,0.207317
3,LSTM TL,0.944535,0.917763,0.968750,0.942568
4,GRU Baseline,0.557912,1.000000,0.059028,0.111475
5,GRU TL,0.978793,0.985866,0.968750,0.977233
6,CNN-LSTM Baseline,0.415987,0.238806,0.111111,0.151659
7,CNN-LSTM TL,0.949429,0.944637,0.947917,0.946274
8,CNN-GRU Baseline,0.415987,0.192982,0.076389,0.109453
9,CNN-GRU TL,0.941272,0.931507,0.944444,0.937931


In [ ]:
save_results_table(results_df, PLOT_PATH)